<a href="https://colab.research.google.com/github/rahmatnug/capstone-cangkringan-ml/blob/main/Generator_Mock_Synthetic_Data_Penjualan_%26_Stok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Set seed untuk reproducibility
np.random.seed(42)

# 1. Master Data Komoditas (6 SKU Riil CV Pandawa Kencana)
komoditas_data = {
    'id_komoditas': [1, 2, 3, 4, 5, 6],
    'nama_komoditas': ['GB Propunic', 'GB Profeed', 'GB Proquatic', 'Pendawa Subur POC', 'Compossap', 'Agen Hayati'],
    'satuan': ['Liter', 'Liter', 'Liter', 'Liter', 'Zak', 'Kg'],
    'base_price': [30000, 30000, 30000, 35000, 25000, 27500],
    'yearly_target': [10000, 9000, 6000, 5500, 3000, 1500] # Kalibrasi dari rekap Excel
}
df_komoditas = pd.DataFrame(komoditas_data)

# 2. Master Data Poktan (Asumsi 5 Poktan untuk distribusi offline)
df_poktan = pd.DataFrame({'id_poktan': [1, 2, 3, 4, 5]})

# 3. Fungsi Helper Cuaca & Musim
def get_musim(bulan):
    if bulan in [11, 12, 1, 2]: return 'Rendeng'
    elif bulan in [3, 4, 5, 6]: return 'Gadu'
    else: return 'Bera'

def get_curah_hujan(musim):
    if musim == 'Rendeng': return round(np.random.uniform(200.0, 450.0), 2)
    elif musim == 'Gadu': return round(np.random.uniform(10.0, 80.0), 2)
    else: return round(np.random.uniform(50.0, 150.0), 2)

# 4. Generate Time-Series Demand (Mingguan: Jan 2025 - Agt 2026)
start_date = datetime(2025, 1, 1)
end_date = datetime(2026, 8, 31)

permintaan_list = []
current_date = start_date
id_permintaan = 1

print("Generating dataset...")
while current_date <= end_date:
    musim = get_musim(current_date.month)
    hujan = get_curah_hujan(musim)

    for p_id in df_poktan['id_poktan']:
       for _, row in df_komoditas.iterrows():
            k_id = row['id_komoditas']
            base_p = row['base_price']
            ytarget = row['yearly_target']

            # Hitung rata-rata volume mingguan per poktan agar total tahunan logis
            avg_weekly = ytarget / 52 / len(df_poktan)

            # Tambahkan pengali tren berdasarkan curah hujan biar polanya sangat jelas
            trend_hujan = hujan / 150.0

            # Kalikan avg_weekly dengan trend_hujan dan pakai noise 5%
            volume = max(1.0, round(np.random.normal(avg_weekly * trend_hujan, avg_weekly * 0.05), 2))

            harga = round(base_p * np.random.uniform(0.9, 1.1), 2) # Fluktuasi harga +/- 10%

            permintaan_list.append({
                'id_permintaan': id_permintaan,
                'id_poktan': p_id,
                'id_komoditas': k_id,
                'tanggal_permintaan': current_date.strftime('%Y-%m-%d'),
                'volume_permintaan': volume,
                'harga_satuan_transaksi': harga,
                'fase_musim': musim,
                'curah_hujan_mm': hujan,
                'status_data': 'Cleaned'
            })
            id_permintaan += 1

    current_date += timedelta(days=7)

df_final = pd.DataFrame(permintaan_list)

# Ekspor ke CSV
df_final.to_csv('synthetic_demand_data.csv', index=False)
print(f"✅ Selesai! File synthetic_demand_data.csv berhasil di-update dengan {len(df_final)} baris dan 6 SKU baru.")

Generating dataset...
✅ Selesai! File synthetic_demand_data.csv berhasil di-update dengan 2610 baris dan 6 SKU baru.
